
# AGN archetypes: Seyfert, quasar, and LIRG/Sy across bolometric luminosity

Three distinct AGN types overlaid to show how AGN morphology and obscuration
evolve with luminosity:

1. **Type-2 obscured Seyfert** (log L_bol ~ 44.5): Heavy obscuration
   dominates; torus reprocesses the central UV/optical into MIR; the
   hot inner disc is hidden; NLR broad lines attenuated.

2. **Type-1 unobscured quasar** (log L_bol ~ 46.5): Bright hot disc
   (big blue bump) fully visible; broad emission lines prominent; minimal
   dust attenuation; radiates across UV to IR with hard X-ray power law.

3. **LIRG/Sy intermediate** (log L_bol ~ 45.5): Moderate starburst–AGN
   blend; mixed obscuration; both torus and starburst dust reprocess;
   narrow-line region visible; bridging the Seyfert–quasar continuum.

This archetype figure is the diagnostic for understanding how AGN

classification depends on viewing angle, accretion rate, and dust geometry.


In [ ]:
import os

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"  # suppress XLA/PjRt C++ INFO+WARNING logs

import warnings

import jax
import matplotlib.pyplot as plt
import numpy as np

import tengri
from tengri.analysis.plotting import setup_style

setup_style()
warnings.filterwarnings("ignore", message=".*BakedInBackend.*")

C_AA_PER_S = 2.998e18

# Use bare-stellar SSP for Cue compatibility (if nebular is added later)
ssp = tengri.load_ssp()

COMMON = dict(
    redshift=tengri.Fixed(0.0),  # Rest-frame only (pred.rest_sed())
)


def build_agn_archetype(log_lbol, agn_lum_ratio, agn_blocks, sfr_log, dust_config):
    """Build an AGN archetype model.

    Parameters
    ----------
    log_lbol : float
        Bolometric AGN luminosity, log10(L_bol / L_sun).
    agn_lum_ratio : float
        AGN fraction of total luminosity.
    agn_blocks : dict
        Composable AGN dict with 'disc', 'torus', 'nlr', 'blr', 'feii', 'atten'.
    sfr_log : float
        log10(SFR / M_sun yr^-1); -10 for pure AGN, positive for starburst blend.
    dust_config : dict
        Dust block dict with 'type', 'tau_diff', 'tau_bc', 'emission'.

    Returns
    -------
    wave : ndarray, shape (n_wave,)
        Rest-frame wavelengths [Angstrom].
    nu_l_nu : ndarray, shape (n_wave,)
        nu * L_nu [erg/s].
    """
    agn_dict = {
        "log_lbol": log_lbol,
        "lum_ratio": agn_lum_ratio,
        "all_params": tengri.FIXED,
    }
    agn_dict.update(agn_blocks)
    # ``sfh.type=const`` is parametrized by total stellar mass over the default
    # 13.8 Gyr window; convert SFR → mass via M = SFR × Δt (+10.14 dex). The
    # ``sfr_log=-10`` "negligible host" cases stay essentially zero; the
    # ``sfr_log=2`` starburst case becomes a realistic ~10^12 M☉ ULIRG host.
    log_total_mass = sfr_log + 10.14
    model = tengri.SEDModel.build(
        ssp,
        sfh={"type": "const", "all_params": tengri.FIXED, "log_total_mass": log_total_mass},
        dust=dust_config,
        agn=agn_dict,
        **COMMON,
    )
    p = dict(model.spec.sample(jax.random.PRNGKey(0)))
    out = model.predict(p)
    wave = np.asarray(model.wavelengths)
    return wave, np.asarray(out.rest_sed())


# ============================================================================
# Archetype 1: Type-2 Obscured Seyfert (log L_bol ~ 44.5)
# ============================================================================
# Heavy torus attenuation (tau_bc ~ 3.0 in front of disc);
# faint visible disc, bright MIR torus re-emission;
# NLR present but dust-attenuated.

wave_sy2, sed_sy2 = build_agn_archetype(
    log_lbol=11.5,
    agn_lum_ratio=1.0,
    agn_blocks=dict(
        disc={"type": "multicolor", "all_params": tengri.FIXED},
        torus={"type": "skirtor", "all_params": tengri.FIXED},
        nlr={"type": "analytic", "all_params": tengri.FIXED},
        blr={"type": "none", "all_params": tengri.FIXED},
    ),
    sfr_log=-10.0,  # Pure AGN, negligible starburst
    dust_config={
        "type": "two_component",
        "tau_diff": 0.1,  # Minimal diffuse dust
        "tau_bc": 3.0,  # Heavy birth-cloud attenuation in front of AGN
        "all_params": tengri.FIXED,
        "emission": {"type": "dale2014", "all_params": tengri.FIXED},
    },
)
nu_l_nu_sy2 = C_AA_PER_S / wave_sy2 * sed_sy2

# ============================================================================
# Archetype 2: Type-1 Unobscured Quasar (log L_bol ~ 46.5)
# ============================================================================
# Bare disc + broad lines fully visible; minimal dust extinction;
# big blue bump dominates optical/UV; hard X-ray power law.

wave_q, sed_q = build_agn_archetype(
    log_lbol=13.5,
    agn_lum_ratio=1.0,
    agn_blocks=dict(
        disc={"type": "multicolor", "all_params": tengri.FIXED},
        torus={"type": "skirtor", "all_params": tengri.FIXED},
        nlr={"type": "none", "all_params": tengri.FIXED},
        blr={"type": "analytic", "all_params": tengri.FIXED},  # BLR instead of NLR
    ),
    sfr_log=-10.0,  # Pure AGN
    dust_config={
        "type": "two_component",
        "tau_diff": 0.0,  # No diffuse dust
        "tau_bc": 0.0,  # No birth-cloud attenuation
        "all_params": tengri.FIXED,
        "emission": {"type": "dale2014", "all_params": tengri.FIXED},
    },
)
nu_l_nu_q = C_AA_PER_S / wave_q * sed_q

# ============================================================================
# Archetype 3: LIRG/Sy Intermediate (log L_bol ~ 45.5)
# ============================================================================
# Moderate AGN + starburst blend; mixed obscuration from torus + diffuse dust;
# visible disc and NLR; intermediate FIR luminosity.

wave_lirg, sed_lirg = build_agn_archetype(
    log_lbol=12.5,
    agn_lum_ratio=0.6,  # Significant starburst contribution
    agn_blocks=dict(
        disc={"type": "multicolor", "all_params": tengri.FIXED},
        torus={"type": "skirtor", "all_params": tengri.FIXED},
        nlr={"type": "analytic", "all_params": tengri.FIXED},
        blr={"type": "none", "all_params": tengri.FIXED},
    ),
    sfr_log=2.0,  # ~100 M_sun / yr ongoing starburst
    dust_config={
        "type": "two_component",
        "tau_diff": 1.5,  # Moderate diffuse dust from starburst
        "tau_bc": 1.0,  # Moderate birth-cloud attenuation
        "all_params": tengri.FIXED,
        "emission": {"type": "dale2014", "all_params": tengri.FIXED},
    },
)
nu_l_nu_lirg = C_AA_PER_S / wave_lirg * sed_lirg

# ============================================================================
# Plot: Overlay all three archetypes
# ============================================================================

fig, ax = plt.subplots(figsize=(8.0, 5.6))

# Use a common wavelength grid for alignment (quasar's grid as reference)
ax.loglog(
    wave_sy2,
    nu_l_nu_sy2,
    color="#d62728",
    lw=1.8,
    label="Type-2 Seyfert (log L$_{\\rm bol}$  = 11.5)",
    alpha=0.8,
)
ax.loglog(
    wave_lirg,
    nu_l_nu_lirg,
    color="#ff7f0e",
    lw=1.8,
    label="LIRG/Sy intermediate (12.5)",
    alpha=0.8,
)
ax.loglog(
    wave_q,
    nu_l_nu_q,
    color="#1f77b4",
    lw=1.8,
    label="Type-1 unobscured quasar (13.5)",
    alpha=0.8,
)

ax.set(
    xlabel=r"Rest-frame wavelength $\lambda$ [$\mathrm{\AA}$]",
    ylabel=r"$\nu L_\nu$  [erg s$^{-1}$]",
    xlim=(100, 1e8),
    ylim=(1e42, 1e48),
)

ax.legend(frameon=False, fontsize=9, loc="lower left")

# Annotate key regions
ax.axvspan(900, 3500, alpha=0.05, color="blue", label="_")
ax.text(
    1500,
    3e47,
    "optical/UV\n(attenuation sensitive)",
    fontsize=7.5,
    color="0.4",
    ha="center",
    va="top",
)

ax.axvspan(1e4, 1e6, alpha=0.05, color="red", label="_")
ax.text(
    5e4,
    5e46,
    "MIR/FIR\n(torus + starburst)",
    fontsize=7.5,
    color="0.4",
    ha="center",
    va="top",
)

fig.tight_layout()
plt.savefig("plot_seyfert_quasar_blazar_archetypes.png", dpi=150, bbox_inches="tight")